### setting up an agent brain

In [68]:
from ultralytics import YOLO
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
from dotenv import load_dotenv
import os
from doc_string import prompt

load_dotenv()  # Load environment variables from .env file

True

In [69]:
ollama_model_client = OllamaChatCompletionClient(
   model="qwen2.5:3b"
   )

In [70]:
def yolo_tool(path: str) -> str:
   """
    Analyze an image using YOLOv8 object detection.

    This tool takes the path of an image, runs YOLOv8 object detection,
    identifies the objects present in the image, and returns the detected
    objects along with their confidence scores and bounding box information.

    Args:
        image_path (str): The file path of the image to analyze.

    Returns:
        str: A summary of the objects detected in the image, including
        their class names and confidence scores.
   """
   model = YOLO("yolov8n.pt") 
   result = model(source=path, show=True)  # predict on an image and show the results
   print(result[0].show()) 
   return "YOLO analysis complete"

In [71]:
yolo_agents = AssistantAgent(
    name="YOLO_Agent",
    model_client=ollama_model_client,
    system_message="You are a helpful assistant that can analyze images using YOLOv8 and provide insights based on the detected objects.",
    tools = [yolo_tool],
    reflect_on_tool_use=True,
    description="This agent can analyze images using YOLOv8 and provide insights based on the detected objects."
) 

In [72]:
x = await yolo_agents.run(task="who are you")

print(x.messages[-1].content)

I am a tool designed to analyze images using YOLOv8 and provide insights based on the detected objects. How can I assist you today?


In [73]:
response = yolo_tool("L:\\agentic-yolo\images\img-1.webp")
print(response)

<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:1: SyntaxWarning: invalid escape sequence '\i'
C:\Users\JILLA SRI VARDHAN\AppData\Local\Temp\ipykernel_3348\665482810.py:1: SyntaxWarning: invalid escape sequence '\i'
  response = yolo_tool("L:\\agentic-yolo\images\img-1.webp")


image 1/1 L:\agentic-yolo\images\img-1.webp: 448x640 5 persons, 1 bench, 3 chairs, 45.8ms
Speed: 2.3ms preprocess, 45.8ms inference, 0.8ms postprocess per image at shape (1, 3, 448, 640)
None
YOLO analysis complete


### teams

In [74]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination 

In [75]:
team = RoundRobinGroupChat(
   participants=[yolo_agents],
   max_turns=3,
   termination_condition=TextMentionTermination("goodbye")
)

In [76]:
from autogen_agentchat.messages import MultiModalMessage
from autogen_core import Image
from PIL import Image as PILImage
from autogen_agentchat.ui import Console

In [77]:
pil_image = PILImage.open("images/img-2.webp")

img = Image(pil_image)

In [78]:
multi_modal_message = MultiModalMessage(
   content=["""Here is an image for analysis.images/img-2.webp""",img],
   source="user"
)

In [79]:
result = await Console(
    team.run_stream(task=multi_modal_message),
)

result.messages[-1].content

---------- MultiModalMessage (user) ----------
Here is an image for analysis.images/img-2.webp
<image>
---------- ToolCallRequestEvent (YOLO_Agent) ----------
[FunctionCall(id='0', arguments='{"image_path": "images/img-2.webp"}', name='yolo_tool')]
---------- ToolCallExecutionEvent (YOLO_Agent) ----------
[FunctionExecutionResult(content="1 validation error for yolo_toolargs\npath\n  Field required [type=missing, input_value={'image_path': 'images/img-2.webp'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing", name='yolo_tool', call_id='0', is_error=True)]
---------- TextMessage (YOLO_Agent) ----------
It seems there was an issue with the input provided. The path to the image might not have been correctly specified or the image file might not be accessible. Could you please provide the correct path to the image again? If you're using a local image, ensure that the path is correct and the file is not corrupted. If you're using an online or 

'Could you please share the correct path to the image?'